# Step 08: Model Explainability — SHAP Global & Local Feature Analysis

## Overview
This notebook computes SHAP (SHapley Additive exPlanations) values for the trained Logistic Regression model:
1. **Global Importance**: Company-wide feature drivers of attrition.
2. **Local Explanation**: Employee-level drill-down extracting top 3 positive & negative risk factors for a sample employee.


In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import shap

from sklearn.model_selection import train_test_split

PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR = os.path.join("..", "models")

pipeline = joblib.load(os.path.join(MODELS_DIR, "attrition_pipeline.joblib"))
df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_engineered.csv"))

drop_cols = ['EmployeeNumber', 'Employee ID', 'Attrition', 'Target_Attrition']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df['Target_Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = pipeline.named_steps['preprocessor']
classifier = pipeline.named_steps['classifier']

# Transform test features
X_test_trans = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("✔ Model & Preprocessor Loaded Successfully.")
print(f"Total transformed features count: {len(feature_names)}")


✔ Model & Preprocessor Loaded Successfully.
Total transformed features count: 52


---
## 1. SHAP Global Feature Importance


In [2]:
explainer = shap.LinearExplainer(classifier, X_test_trans)
shap_values = explainer.shap_values(X_test_trans)

# Calculate mean absolute SHAP value per feature
mean_shap = np.abs(shap_values).mean(axis=0)
global_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Absolute_SHAP': mean_shap
}).sort_values(by='Mean_Absolute_SHAP', ascending=False)

print("=== Top 10 Company-Wide Attrition Drivers (SHAP Global) ===")
print(global_importance_df.head(10).to_string(index=False))


Background dataset has 294 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=294 when initializing the masker.


=== Top 10 Company-Wide Attrition Drivers (SHAP Global) ===
                              Feature  Mean_Absolute_SHAP
                    cat__OverTime_Yes            0.649268
               num__TotalWorkingYears            0.595877
cat__BusinessTravel_Travel_Frequently            0.462770
         num__YearsSinceLastPromotion            0.451531
   cat__JobRole_Laboratory Technician            0.427490
                        num__JobLevel            0.402579
            cat__MaritalStatus_Single            0.401508
    cat__BusinessTravel_Travel_Rarely            0.361448
              num__NumCompaniesWorked            0.357633
              num__YearsInCurrentRole            0.289387


---
## 2. SHAP Local Explanation (Individual Employee Risk Drivers)


In [3]:
# Select a sample high-risk employee from test set
high_risk_indices = np.where((classifier.predict_proba(X_test_trans)[:, 1] > 0.6))[0]
sample_idx = high_risk_indices[0] if len(high_risk_indices) > 0 else 0

emp_shap = shap_values[sample_idx]
emp_proba = classifier.predict_proba(X_test_trans[sample_idx:sample_idx+1])[0, 1]

local_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP_Value': emp_shap
}).sort_values(by='SHAP_Value', ascending=False)

print(f"=== Local SHAP Explanation for Test Employee #{sample_idx} ===")
print(f"Predicted Attrition Risk Probability: {emp_proba*100:.1f}%")
print("\nTop 3 Factors INCREASING Attrition Risk:")
print(local_df.head(3).to_string(index=False))
print("\nTop 3 Factors REDUCING Attrition Risk:")
print(local_df.tail(3).to_string(index=False))


=== Local SHAP Explanation for Test Employee #4 ===
Predicted Attrition Risk Probability: 74.2%

Top 3 Factors INCREASING Attrition Risk:
                   Feature  SHAP_Value
         cat__OverTime_Yes    1.283979
    num__TotalWorkingYears    0.888200
num__TrainingTimesLastYear    0.435533

Top 3 Factors REDUCING Attrition Risk:
                     Feature  SHAP_Value
num__YearsSinceLastPromotion   -0.393520
               num__JobLevel   -0.477250
     num__NumCompaniesWorked   -0.499144
